# Imports


In [1]:
# Paths / folders
from pathlib import Path
raw_data_path = "../data/gl/01_raw/*.csv"
silver_data_path = "../data/gl/02_silver"
metadata_path = "../data/gl/metadata"

Path(silver_data_path).mkdir(parents=True, exist_ok=True)
Path(metadata_path).mkdir(parents=True, exist_ok=True)

# PySpark imports
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, 
    expr, 
    regexp_replace, 
    regexp_extract, 
    input_file_name
)
from pyspark.sql.types import (
    StructType, 
    StructField, 
    StringType, 
    IntegerType, 
    DoubleType
)

# Data Processing

### Initial Data Loading / Schema Mapping

In [2]:
# Spark session
spark = (
    SparkSession.builder.appName("gl_transform")
    .master("local[*]")
    .config("spark.driver.memory", "16g")
    .config("spark.sql.shuffle.partitions", "32")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.minPartitionSize", "128MB")
    .getOrCreate()
)

# Set Spark log level to ERROR
spark.sparkContext.setLogLevel("ERROR")
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/22 18:38:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/02/22 18:38:39 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
# Dataset schema
schema = StructType(
    [
        StructField("AGENCYNBR", StringType(), True),
        StructField("AGENCYNAME", StringType(), True),
        StructField("LEDGER", StringType(), True),
        # - FISCAL_YEAR and ACCOUNTING_PERIOD need some downstream cleaning
        StructField("FISCAL_YEAR", StringType(), True),
        StructField("ACCOUNTING_PERIOD", StringType(), True),
        StructField("FUND_CODE", StringType(), True),
        StructField("FUNDDESCR", StringType(), True),
        StructField("CLASS_FLD", StringType(), True),
        StructField("CLASSDESCR", StringType(), True),
        StructField("DEPTID", IntegerType(), True),
        StructField("DEPTDESCR", StringType(), True),
        StructField("ACCOUNT", IntegerType(), True),
        StructField("ACCTDESCR", StringType(), True),
        StructField("OPERATING_UNIT", StringType(), True),
        StructField("OPERUNITDESCR", StringType(), True),
        StructField("PRODUCT", StringType(), True),
        StructField("PRODUCTDESCR", StringType(), True),
        StructField("PROGRAM_CODE", StringType(), True),
        StructField("PGMDESCR", StringType(), True),
        StructField("BUDGET_REF", StringType(), True),
        StructField("CHARTFIELD1", StringType(), True),
        StructField("CF1DESCR", StringType(), True),
        StructField("CHARTFIELD2", StringType(), True),
        StructField("CF2DESCR", StringType(), True),
        StructField("PROJECT_ID", StringType(), True),
        StructField("PROJDESCR", StringType(), True),
        StructField("POSTED_TOTAL_AMT", DoubleType(), True),
        StructField("ACTIVITY", StringType(), True),
        StructField("ACTVDESCR", StringType(), True),
        StructField("RESTYPE", StringType(), True),
        StructField("RESDESCR", StringType(), True),
        StructField("RCAT", StringType(), True),
        StructField("RCATDESCR", StringType(), True),
        StructField("RSUBCAT", StringType(), True),
        StructField("RSUBCATDESCR", StringType(), True),
        StructField("ROWID", StringType(), True),
    ]
)

# Read data from raw
df = (
    spark.read
    .option("header", True)
    .schema(schema)
    .csv(raw_data_path)
    .withColumn("source_file", input_file_name())
)

df.show(5)

+---------+-----------------+-------+--------------------+--------------------+---------+--------------------+---------+--------------------+-------+--------------------+-------+--------------------+--------------+-------------+-------+------------+------------+--------+----------+-----------+--------+-----------+--------+----------+---------+----------------+--------+---------+-------+--------+----+---------+-------+------------+------------------+--------------------+
|AGENCYNBR|       AGENCYNAME| LEDGER|         FISCAL_YEAR|   ACCOUNTING_PERIOD|FUND_CODE|           FUNDDESCR|CLASS_FLD|          CLASSDESCR| DEPTID|           DEPTDESCR|ACCOUNT|           ACCTDESCR|OPERATING_UNIT|OPERUNITDESCR|PRODUCT|PRODUCTDESCR|PROGRAM_CODE|PGMDESCR|BUDGET_REF|CHARTFIELD1|CF1DESCR|CHARTFIELD2|CF2DESCR|PROJECT_ID|PROJDESCR|POSTED_TOTAL_AMT|ACTIVITY|ACTVDESCR|RESTYPE|RESDESCR|RCAT|RCATDESCR|RSUBCAT|RSUBCATDESCR|             ROWID|         source_file|
+---------+-----------------+-------+-------------

## Transform
- Clean `FISCAL_YEAR` and `ACCOUNTING_PERIOD`
- Columns that are wholly null (drop these all (explore dynamic detection of these in code later)):
  - `['PGMDESCR', 'OPERUNITDESCR', 'RSUBCATDESCR', 'RSUBCAT', 'RCATDESCR',
      'RCAT', 'RESDESCR', 'RESTYPE', 'ACTVDESCR', 'CF1DESCR', 'CF2DESCR',
       'PROJDESCR', 'PRODUCTDESCR']`

In [4]:
# Clean FISCAL_YEAR and ACCOUNTING_PERIOD (decimal and 0's)
df = (
    df
    .withColumn(
        "FISCAL_YEAR", 
        expr("try_cast(regexp_replace(FISCAL_YEAR, '\\\\.0+$', '') as int)")
    )
    .withColumn(
        "ACCOUNTING_PERIOD", 
        expr("try_cast(regexp_replace(ACCOUNTING_PERIOD, '\\\\.0+$', '') as int)")
    )
    .withColumn(
        "EXTRACTED_QUARTER", regexp_extract(col("source_file"), r"QTR(\d)", 1)
    )
)

# Extract the quarter from filename
df = df.withColumn("EXTRACTED_QUARTER", regexp_extract(col("source_file"), r"QTR(\d)", 1))

# Preview
df.printSchema()
df.show(5)

root
 |-- AGENCYNBR: string (nullable = true)
 |-- AGENCYNAME: string (nullable = true)
 |-- LEDGER: string (nullable = true)
 |-- FISCAL_YEAR: integer (nullable = true)
 |-- ACCOUNTING_PERIOD: integer (nullable = true)
 |-- FUND_CODE: string (nullable = true)
 |-- FUNDDESCR: string (nullable = true)
 |-- CLASS_FLD: string (nullable = true)
 |-- CLASSDESCR: string (nullable = true)
 |-- DEPTID: integer (nullable = true)
 |-- DEPTDESCR: string (nullable = true)
 |-- ACCOUNT: integer (nullable = true)
 |-- ACCTDESCR: string (nullable = true)
 |-- OPERATING_UNIT: string (nullable = true)
 |-- OPERUNITDESCR: string (nullable = true)
 |-- PRODUCT: string (nullable = true)
 |-- PRODUCTDESCR: string (nullable = true)
 |-- PROGRAM_CODE: string (nullable = true)
 |-- PGMDESCR: string (nullable = true)
 |-- BUDGET_REF: string (nullable = true)
 |-- CHARTFIELD1: string (nullable = true)
 |-- CF1DESCR: string (nullable = true)
 |-- CHARTFIELD2: string (nullable = true)
 |-- CF2DESCR: string (nulla

In [5]:
# Columns to drop
null_columns = [
   'PGMDESCR', 'OPERUNITDESCR', 'RSUBCATDESCR', 'RSUBCAT', 'RCATDESCR',
    'RCAT', 'RESDESCR', 'RESTYPE', 'ACTVDESCR', 'CF1DESCR', 'CF2DESCR',
    'PROJDESCR', 'PRODUCTDESCR' 
]

# Drop these columns and keep a record
df = df.drop(*null_columns)

with open(f"{metadata_path}/historical_backfill_dropped_columns_log.txt", "w") as f:
    f.write("\n".join(null_columns) + "\n")


df.show(5)

+---------+-----------------+-------+-----------+-----------------+---------+--------------------+---------+--------------------+-------+--------------------+-------+--------------------+--------------+-------+------------+----------+-----------+-----------+----------+----------------+--------+------------------+--------------------+-----------------+
|AGENCYNBR|       AGENCYNAME| LEDGER|FISCAL_YEAR|ACCOUNTING_PERIOD|FUND_CODE|           FUNDDESCR|CLASS_FLD|          CLASSDESCR| DEPTID|           DEPTDESCR|ACCOUNT|           ACCTDESCR|OPERATING_UNIT|PRODUCT|PROGRAM_CODE|BUDGET_REF|CHARTFIELD1|CHARTFIELD2|PROJECT_ID|POSTED_TOTAL_AMT|ACTIVITY|             ROWID|         source_file|EXTRACTED_QUARTER|
+---------+-----------------+-------+-----------+-----------------+---------+--------------------+---------+--------------------+-------+--------------------+-------+--------------------+--------------+-------+------------+----------+-----------+-----------+----------+----------------+------

## Write to Silver
- Partition by `FISCAL_YEAR`

In [6]:
# Partitioning by FISCAL_YEAR and ACCOUNTING_PERIOD creates very small files
# For now, use FISCAL_YEAR only
df.repartition("FISCAL_YEAR").write\
    .mode("overwrite")\
    .partitionBy("FISCAL_YEAR")\
    .parquet(f"{silver_data_path}/gl")